# Вес ключей в принятой награде (копия `reward_perturbations.ipynb`)

Решение 08.11.2026: проверить заранее, не снизить ли вес ключей. Принятая награда (`RewardWeights()` по умолчанию:
bge-m3, α = 0,2, прямые штрафы, связь визуала со слайдом, числа и единицы, повтор визуала — лишний элемент, цена вызова
0,003) при весе ключей 0; 0,25; 0,5 (текущий); 1, λ = 1. Те же пары и правила, что в EXP-014 и EXP-016 (тетрадка —
копия, исходная не меняется). Смотрим: насколько при меньшем весе слабеют «неверное число» и «удалённое объяснение»,
насколько уменьшается лазейка «список ключей», что говорит разметка пар с конфликтом; то же — на новых исходниках.
Результат — `runs/perturbation/<дата>/` (`ablation.csv`, `holdout_ablation.csv`, `keys_conflicts.csv`).


In [1]:
import csv, json, os, statistics, sys, time
from dataclasses import replace
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # перед запуском проверить nvidia-smi
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl import perturb_eval as pe
from presentation_rl.rewards import Lemmatizer, load_key_points
from presentation_rl.tasks import load_tasks

OUT = ROOT / "runs" / "perturbation" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "runs" / "perturbation" / "embeddings"  # кеш эмбеддингов общий для всех запусков
EMBEDDERS = ["intfloat/multilingual-e5-small", "BAAI/bge-m3"]
AXES = {"alpha": [0.0, 0.2, 0.4], "length": [0.5, 1.0, 2.0], "keys": [0.5, 1.0]}  # λ — length
RUN = "keys"  # вес ключей в принятой награде; "ablation" — EXP-016, "grid" — EXP-014
ABLATION = ("BAAI/bge-m3", 0.2)  # эмбеддер и α второго прогона (решение 07.11.2026)
SFT_PLAIN = ROOT / "data" / "tasks" / "sft" / "sft_dataset.jsonl"  # чужие презентации для фона тезисов
KEYS = [0.0, 0.25, 0.5, 1.0]  # веса ключей при λ = 1
if RUN == "keys":
    EMBEDDERS = [ABLATION[0]]
    CURRENT = pe.Config(*ABLATION, *pe.CURRENT_WEIGHTS, "accepted")  # принятая награда, вес ключей 0,5
elif RUN == "ablation":
    EMBEDDERS = [ABLATION[0]]
    CURRENT = pe.Config(*ABLATION, *pe.CURRENT_WEIGHTS)  # исходная награда на bge-m3, α = 0,2
else:
    CURRENT = pe.Config(EMBEDDERS[0], 0.0, *pe.CURRENT_WEIGHTS)  # текущие коэффициенты (RewardWeights)
RUNS = [ROOT / "runs" / "variance" / "20261020_0331", ROOT / "runs" / "reward_sources" / "20261027_2150"]
M_FACTORS = [0.15, 0.25, 0.5]  # основной — 0,25 (pe.M_SHARE); остальные — чувствительность
RULES = pe.Rules()
CODEX_KINDS = ["repeat_vs_missing"]  # где Codex дополняет автора
KEYS_MIN = 10  # меньше решённых пар про ключи — вес ключей не выбирается
print(OUT)

mkdir -p failed for path /home/jovyan/.config/matplotlib: [Errno 20] Not a directory: '/home/jovyan/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-rri9u06s because there was an issue with the default path (/home/jovyan/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


/home/jovyan/Diplom/runs/perturbation/20261110_1253


In [2]:
def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        return
    fields = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="", encoding="utf-8-sig") as f:  # utf-8-sig — для Excel
        writer = csv.DictWriter(f, fields)
        writer.writeheader()
        writer.writerows(rows)


points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json")}
items, pairs = pe.load_variants(ROOT / "data" / "perturbation" / "variants")
contexts = {key[0]: item.context for key, item in items.items() if key[1] == "original"}
MODES = {t: ("search" if c["search"] else "plain") for t, c in contexts.items()}
natural = pe.load_natural(RUNS, tasks, set(contexts))
conflict_items, conflicts = pe.load_conflicts(ROOT / "data" / "perturbation" / "conflicts", contexts)
all_items = {**items, **natural, **conflict_items}
print(f"вариантов {len(items)}, пар {len(pairs)}, ответов T-lite {len(natural)}, презентаций из пар с конфликтом "
      f"{len(conflict_items)}")
start = time.time()
pe.build_all(all_items)
pairs, dropped = pe.check_pairs(pairs, items)
print(f"сборка {time.time() - start:.0f} с; пар после предусловий {len(pairs)}, отброшено {len(dropped)}")
for line in dropped:
    print("  -", line)

вариантов 1099, пар 1099, ответов T-lite 256, презентаций из пар с конфликтом 160
собрано 200 из 1515
собрано 400 из 1515
собрано 600 из 1515
собрано 800 из 1515
собрано 1000 из 1515
собрано 1200 из 1515
собрано 1400 из 1515
сборка 415 с; пар после предусловий 1099, отброшено 0


## Судья для пар с конфликтом

Решение 01.11.2026 (`decisions.md`, раздел 7): судья — автор (50 пар, разметка вслепую); на парах «повторы против
отсутствия заключения», которые автор не размечал, — Codex (с автором на этом виде совпал в 8 из 10). Ответы
«равны» и пары, где Codex в двух порядках выбрал разное, не учитываются. Вес ключей выбирается по парам
с конфликтом, только если решённых пар про ключи не меньше 10; иначе — текущий (0,5).


In [3]:
pe.combine_judges(conflicts, "author", "codex", CODEX_KINDS, "judge")
JUDGE = "judge"
for kind in sorted({c["kind"] for c in conflicts}):
    part = [c for c in conflicts if c["kind"] == kind]
    print(f"  {kind}: решено автором {sum(c['judges'].get('author') not in (None, 'equal') for c in part)}, "
          f"сводным судьёй {sum(c['judges'].get('judge') not in (None, 'equal') for c in part)} из {len(part)}; "
          f"Codex определился в {sum(c['judges'].get('codex') is not None for c in part)}")
same, both = pe.judges_agreement(conflicts, "codex", "author")
print(f"согласие Codex и автора (второе мнение, для отчёта): {same} из {both}")
decided = {d: sum(c["judges"].get(JUDGE) not in (None, "equal") for c in conflicts if c["decides"] == d)
           for d in ("length", "keys")}
CHOOSE_KEYS = decided["keys"] >= KEYS_MIN
print("решённых пар:", decided, "| вес ключей", "выбирается" if CHOOSE_KEYS else "остаётся текущим (мало пар)")


  repeat_vs_missing: решено автором 10, сводным судьёй 16 из 16; Codex определился в 16
  stretch_all_vs_missing1: решено автором 10, сводным судьёй 10 из 16; Codex определился в 15
  stretch_all_vs_missing2: решено автором 8, сводным судьёй 8 из 16; Codex определился в 15
  stretch_half_vs_missing1: решено автором 12, сводным судьёй 12 из 16; Codex определился в 14
  text_vs_keys: решено автором 14, сводным судьёй 14 из 16; Codex определился в 14
согласие Codex и автора (второе мнение, для отчёта): 24 из 48
решённых пар: {'length': 46, 'keys': 14} | вес ключей выбирается


In [4]:
if RUN == "grid":
    configs = pe.grid(EMBEDDERS, AXES["alpha"], AXES["length"], AXES["keys"])
elif RUN == "keys":
    configs = [pe.Config(*ABLATION, pe.CURRENT_WEIGHTS[0], k, "accepted") for k in KEYS]
else:
    configs = pe.ablation_configs(*ABLATION, AXES["length"], AXES["keys"])
foreign = pe.load_foreign(SFT_PLAIN) if RUN == "ablation" else None
originals = {key[0]: item.deck for key, item in items.items() if key[1] == "original"}
domains = {t: tasks[t].request.domain for t in originals}
TAU = {}  # пороги визуала по эмбеддерам (нужны и для новых исходников)


def run_grid(configs):
    lemmatizer = Lemmatizer()
    scores, sigmas = {}, {}
    for name in sorted({c.embedder for c in configs}):
        start = time.time()
        embedder = pe.CachedEmbedder(name, CACHE)
        print(name, embedder.signature, "в кеше", len(embedder.vectors))
        tau = None
        if any(c.variant != "base" for c in configs if c.embedder == name):
            tau, tau_report = pe.calibrate_tau(originals, domains, embedder)
            TAU[name] = tau
            write_csv(OUT / f"tau_{name.split('/')[-1]}.csv", tau_report)
            print("  пороги визуала:", tau, "| без порога (половина бонуса):",
                  [r["kind"] for r in tau_report if not r["verified"]])
        for config in [c for c in configs if c.embedder == name]:
            scores[config] = pe.score_items(all_items, points, config, embedder, lemmatizer, foreign, tau)
            sigmas[config], _ = pe.sigma(scores[config])
            embedder.save()
            print(f"  {config.name}: σ = {sigmas[config]:.4f}")
        print(f"  {time.time() - start:.0f} с")
    return scores, sigmas


scores, sigmas = run_grid(configs)


def evaluate(scores, sigmas, m_factor=pe.M_SHARE):
    rows_by, conf_by = {}, {}
    for config, s in scores.items():
        m, eps = m_factor * sigmas[config], pe.EPS_SHARE * sigmas[config]
        rows_by[config] = pe.evaluate_pairs(pairs, s, m, eps)
        conf_by[config] = pe.conflict_rows(conflicts, s, JUDGE, eps) if JUDGE else []
    return rows_by, conf_by


rows_by, conf_by = evaluate(scores, sigmas)
write_csv(OUT / "sigma.csv", ({"config": c.name, **g} for c, s in scores.items() for g in pe.sigma(s)[1]))
write_csv(OUT / "components.csv", ({"config": c.name, "key": "/".join(map(str, k)), **v}
                                   for c, s in scores.items() for k, v in s.items()))
write_csv(OUT / "pairs.csv", ({"config": c.name, **r} for c, rows in rows_by.items() for r in rows))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BAAI/bge-m3 {'model': 'BAAI/bge-m3', 'revision': '5617a9f61b028005a4858fdac845db406aefb181', 'prefix': '', 'max_seq_length': 8192, 'normalize': True} в кеше 8056
  пороги визуала: {'DIAGRAM': 0.5218929052352905, 'IMAGE': 0.4112645089626312} | без порога (половина бонуса): ['CHART', 'PLOT', 'TIMELINE']
  bge-m3|a=0.2|len=1.0|keys=0.0|accepted: σ = 0.0787
  bge-m3|a=0.2|len=1.0|keys=0.25|accepted: σ = 0.0800
  bge-m3|a=0.2|len=1.0|keys=0.5|accepted: σ = 0.0829
  bge-m3|a=0.2|len=1.0|keys=1.0|accepted: σ = 0.0955
  401 с


## Величина при текущих коэффициентах

Сдвиг награды и целевого компонента по видам искажений в сравнении с естественным разбросом σ: штраф должен быть
заметен на фоне σ и не перевешивать содержание.

In [5]:
summary = []
for config, rows in rows_by.items():
    for entry in pe.summarize(rows):
        entry["d_total_median_sigma"] = entry["d_total_median"] / sigmas[config]
        summary.append({"config": config.name, "sigma": sigmas[config], **entry})
write_csv(OUT / "summary.csv", summary)
cols = ["family", "kind", "mode", "n", "d_total_median", "d_total_median_sigma", "d_target_median", "total_ok",
        "total_violation", "component_ok", "component_down_total_up"]
print(f"текущие коэффициенты: {CURRENT.name}, σ = {sigmas[CURRENT]:.4f}")
for e in (s for s in summary if s["config"] == CURRENT.name):
    print("  ".join(f"{e[c]:.3f}" if isinstance(e[c], float) else str(e[c]) for c in cols))
print("провал в одном режиме (предупреждения, без отсева):", pe.mode_failures(rows_by[CURRENT], RULES))

текущие коэффициенты: bge-m3|a=0.2|len=1.0|keys=0.5|accepted, σ = 0.0829
chain  bad_source_chain  plain  8  -0.200  -2.412  -2.000  1.000  0.000  1.000  0
chain  bad_source_chain  search  8  -0.200  -2.412  -2.000  1.000  0.000  1.000  0
chain  extra_calls_chain  search  8  -0.006  -0.072  -0.006  0.000  0.000  0.000  0
chain  fake_quote_chain  search  8  -0.204  -2.466  -2.000  1.000  0.000  1.000  0
chain  untagged_chain  plain  16  -0.003  -0.040  -0.034  0.000  0.000  1.000  0
chain  untagged_chain  search  16  -0.003  -0.041  -0.034  0.000  0.000  1.000  0
coverage  conclusion_empty  plain  8  -0.063  -0.756  -0.039  1.000  0.000  0.875  0
coverage  conclusion_empty  search  7  -0.055  -0.661  -0.036  1.000  0.000  0.857  0
coverage  conclusion_replaced  plain  8  -0.063  -0.757  -0.026  0.875  0.000  0.500  0
coverage  conclusion_replaced  search  7  -0.059  -0.706  -0.013  1.000  0.000  0.429  0
coverage  drop_explanation  plain  20  -0.058  -0.703  -0.022  0.950  0.000  0.700  

## Абляции (второй прогон)

Каждое изменение награды по отдельности и все вместе при текущих λ и весе ключей: медиана Δ награды по видам
искажений в σ исходной награды и доля выполненных ожиданий по старому критерию (пороги от σ исходной награды).
Это регрессия на тех же парах, по которым изменения придуманы; независимая проверка — на новых исходниках.


In [6]:
if RUN in ("ablation", "keys"):
    ablation = pe.ablation_rows(pairs, scores, sigmas, CURRENT)
    write_csv(OUT / "ablation.csv", ablation)
    shown = configs if RUN == "keys" else [c for c in configs if (c.length, c.keys) == pe.CURRENT_WEIGHTS]
    label = (lambda c: f"keys={c.keys}") if RUN == "keys" else (lambda c: c.variant)
    print("σ:", "  ".join(f"{label(c)}={sigmas[c]:.4f}" for c in shown))
    table = {}
    for r in ablation:
        table.setdefault((r["family"], r["kind"], r["mode"]), {})[r["config"]] = r
    print(f"{'вид искажения':42s}" + "".join(f"{label(c)[:9]:>10s}" for c in shown) + "   (медиана Δ в σ текущей / доля ok по её критерию)")
    for key, row in sorted(table.items()):
        cells_ = [row.get(c.name) for c in shown]
        print(f"{'/'.join(map(str, key))[:42]:42s}" + "".join(
            f"{r['d_sigma_base']:>+6.2f}/{r['total_ok_base_criterion'] or 0:.1f}"[:10].rjust(10) if r else " " * 10
            for r in cells_))
    if RUN == "keys" and JUDGE:  # совпадение с судьёй на парах с конфликтом при каждом весе ключей
        keys_conflicts = []
        for c in shown:
            rows = conf_by[c]
            for part, name in ((rows, "все"), ([r for r in rows if r["decides"] == "keys"], "про ключи"),
                               ([r for r in rows if r["decides"] == "length"], "про длину")):
                if part:
                    keys_conflicts.append({"keys": c.keys, "pairs": name, "n": len(part),
                                           "agree": sum(r["agree"] for r in part) / len(part)})
        write_csv(OUT / "keys_conflicts.csv", keys_conflicts)
        for r in keys_conflicts:
            print(f"вес ключей {r['keys']}: пары с конфликтом, {r['pairs']}: совпадение с судьёй {r['agree']:.2f} (n={r['n']})")


σ: keys=0.0=0.0787  keys=0.25=0.0800  keys=0.5=0.0829  keys=1.0=0.0955
вид искажения                               keys=0.0 keys=0.25  keys=0.5  keys=1.0   (медиана Δ в σ текущей / доля ok по её критерию)
chain/bad_source_chain/plain               -2.41/1.0 -2.41/1.0 -2.41/1.0 -2.41/1.0
chain/bad_source_chain/search              -2.41/1.0 -2.41/1.0 -2.41/1.0 -2.41/1.0
chain/extra_calls_chain/search             -0.07/0.0 -0.07/0.0 -0.07/0.0 -0.07/0.0
chain/fake_quote_chain/search              -2.47/1.0 -2.47/1.0 -2.47/1.0 -2.47/1.0
chain/untagged_chain/plain                 -0.04/0.0 -0.04/0.0 -0.04/0.0 -0.04/0.0
chain/untagged_chain/search                -0.04/0.0 -0.04/0.0 -0.04/0.0 -0.04/0.0
coverage/conclusion_empty/plain            -0.54/1.0 -0.64/1.0 -0.76/1.0 -0.95/1.0
coverage/conclusion_empty/search           -0.63/1.0 -0.66/1.0 -0.66/1.0 -0.66/1.0
coverage/conclusion_replaced/plain         -0.56/0.9 -0.64/0.9 -0.76/0.9 -0.98/0.9
coverage/conclusion_replaced/search        -0.68

## Выбор конфигурации

In [7]:
def select_all(rows_by, conf_by, rules=RULES):
    allowed = [c for c in rows_by if CHOOSE_KEYS or c.keys == pe.CURRENT_WEIGHTS[1]]
    tallies = {c: pe.tally(rows_by[c]) for c in allowed}
    ctallies = {c: pe.conflict_tally(conf_by[c]) for c in allowed} if JUDGE else None
    return pe.nested_cv(tallies, ctallies, MODES, rules=rules), tallies


cv, tallies = select_all(rows_by, conf_by)
for fold in cv["outer"]:
    print(fold)
print("итог:", cv["final"], cv["final_info"])
print(f"устойчив: {cv['stable']}, оптимизм: {cv['optimism']}, надёжен: {cv['reliable']}")
final = next(c for c in configs if c.name == cv["final"])
chosen_axes = ["alpha"] + (["length"] + (["keys"] if CHOOSE_KEYS else []) if JUDGE and "kept_current" not in cv["final_info"] else [])
extend = pe.edge(final, configs, chosen_axes)
print("на краю сетки — расширить:", extend or "нет")
if JUDGE:
    print("различают ли пары с конфликтом λ и вес ключей:", pe.identifiable(conf_by))
ci = pe.bootstrap(tallies[final], sorted(MODES))
shares = pe.family_shares(tallies[final], {t: 1 for t in MODES})
write_csv(OUT / "bootstrap.csv", [{"family": f, "share": shares.get(f), "low": lo, "median": med, "high": hi}
                                  for f, (lo, med, hi) in sorted(ci.items())])
(OUT / "cv.json").write_text(json.dumps(cv, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
for f, (lo, med, hi) in sorted(ci.items()):
    print(f"{f:14s} {shares.get(f, float('nan')):.2f}  [{lo:.2f}; {hi:.2f}]")

{'test': ['t380', 't368', 't367', 't375'], 'chosen': 'bge-m3|a=0.2|len=1.0|keys=0.5|accepted', 'test_criterion': -inf, 'train_criterion': 0.8188594666716625, 'test_agreement': 0.7142857142857143, 'train_agreement': 0.7391304347826086}
{'test': ['t369', 't373', 't365', 't374'], 'chosen': 'bge-m3|a=0.2|len=1.0|keys=0.5|accepted', 'test_criterion': -inf, 'train_criterion': 0.8089464170581192, 'test_agreement': 0.8125, 'train_agreement': 0.7045454545454546}
{'test': ['t384', 't389', 't377', 't386'], 'chosen': 'bge-m3|a=0.2|len=1.0|keys=0.5|accepted', 'test_criterion': 0.8360328489540368, 'train_criterion': 0.7994573249809479, 'test_agreement': 0.8125, 'train_agreement': 0.7045454545454546}
{'test': ['t371', 't388', 't381', 't382'], 'chosen': 'bge-m3|a=0.2|len=1.0|keys=0.5|accepted', 'test_criterion': 0.805059081417777, 'train_criterion': 0.8095839738982926, 'test_agreement': 0.5714285714285714, 'train_agreement': 0.782608695652174}
итог: bge-m3|a=0.2|len=1.0|keys=0.5|accepted {'exempt': ['

## Расширение сетки (одно, по правилу)

Если выбранная конфигурация на краю сетки по α, λ или весу ключей — добавить значение за краем и повторить
выбор теми же правилами. Расширение делается один раз.

In [8]:
if extend and RUN == "grid":
    for axis, value in extend.items():
        AXES[axis] = sorted(AXES[axis] + [value])
    extra = [c for c in pe.grid(EMBEDDERS, AXES["alpha"], AXES["length"], AXES["keys"]) if c not in scores]
    more_scores, more_sigmas = run_grid(extra)
    scores.update(more_scores)
    sigmas.update(more_sigmas)
    configs = list(scores)
    rows_by, conf_by = evaluate(scores, sigmas)
    cv, tallies = select_all(rows_by, conf_by)
    final = next(c for c in configs if c.name == cv["final"])
    print("после расширения:", cv["final"], "| устойчив:", cv["stable"], "| надёжен:", cv["reliable"])
    (OUT / "cv_extended.json").write_text(json.dumps(cv, ensure_ascii=False, indent=1, default=str), encoding="utf-8")

## Чувствительность к порогам (для статьи)

Выбор делается только по основным правилам; здесь — меняется ли победитель при других порогах.

In [9]:
variants = []
for m_factor in M_FACTORS:
    rows_m, conf_m = evaluate(scores, sigmas, m_factor)
    for fail_core, fail_other in ((0.6, 0.4), (0.7, 0.5), (0.8, 0.6)):
        for close_scale in (0.5, 1.0, 2.0):
            rules = replace(RULES, fail_core=fail_core, fail_other=fail_other, close_scale=close_scale)
            cv_s, _ = select_all(rows_m, conf_m, rules)
            variants.append({"m": m_factor, "fail_core": fail_core, "fail_other": fail_other, "close": close_scale,
                             "chosen": cv_s["final"], "same_as_main": cv_s["final"] == final.name,
                             "feasible": cv_s["final_info"]["feasible"], "exempt": ";".join(cv_s["final_info"]["exempt"]),
                             "stable": cv_s["stable"], "optimism": cv_s["optimism"]})
write_csv(OUT / "sensitivity.csv", variants)
print(f"победитель тот же в {sum(v['same_as_main'] for v in variants)} из {len(variants)} вариантов порогов")

победитель тот же в 18 из 27 вариантов порогов


## Независимая проверка на новых исходниках

Решение 07.11.2026: изменения награды придуманы по парам 16 исходников, поэтому их проверка на тех же парах — регрессия.
Независимая проверка — на новых исходниках тем validation (`data/perturbation/holdout`, `python -m presentation_rl.perturb
--base data/perturbation/holdout`): те же конфигурации, пороги m и ε — от σ основного набора, по каждой — доля
выполненных ожиданий по семействам. Результат — `holdout_ablation.csv`, `holdout_pairs.csv`.


In [10]:
HOLDOUT = ROOT / "data" / "perturbation" / "holdout" / "variants"
if RUN in ("ablation", "keys") and HOLDOUT.exists():
    hold_items, hold_pairs = pe.load_variants(HOLDOUT)
    pe.build_all(hold_items)
    hold_pairs, hold_dropped = pe.check_pairs(hold_pairs, hold_items)
    print(f"новые исходники: вариантов {len(hold_items)}, пар {len(hold_pairs)}, отброшено {len(hold_dropped)}")
    lemmatizer = Lemmatizer()
    embedder = pe.CachedEmbedder(ABLATION[0], CACHE)
    hold_scores = {c: pe.score_items(hold_items, points, c, embedder, lemmatizer, foreign, TAU.get(c.embedder))
                   for c in configs}
    embedder.save()
    write_csv(OUT / "holdout_ablation.csv", pe.ablation_rows(hold_pairs, hold_scores, sigmas, CURRENT))
    hold_rows = {c: pe.evaluate_pairs(hold_pairs, s, pe.M_SHARE * sigmas[c], pe.EPS_SHARE * sigmas[c])
                 for c, s in hold_scores.items()}
    write_csv(OUT / "holdout_pairs.csv", ({"config": c.name, **r} for c, rows in hold_rows.items() for r in rows))
    hold_weights = {t: 1 for t in {p["task"] for p in hold_pairs}}
    for c in configs:
        shares = pe.family_shares(pe.tally(hold_rows[c]), hold_weights)
        print(f"{c.name:45s} " + "  ".join(f"{f}={v:.2f}" for f, v in sorted(shares.items()))
              + ("  ← выбрана" if c == final else ""))


собрано 200 из 254
новые исходники: вариантов 254, пар 252, отброшено 2


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

bge-m3|a=0.2|len=1.0|keys=0.0|accepted        coverage=0.83  format=0.60  grounding=0.97  invariance=0.92  keys=0.50  length=1.00  text=1.00  tools=0.62  visual=0.94
bge-m3|a=0.2|len=1.0|keys=0.25|accepted       coverage=0.87  format=0.63  grounding=0.97  invariance=0.92  keys=0.52  length=1.00  text=1.00  tools=0.62  visual=0.94
bge-m3|a=0.2|len=1.0|keys=0.5|accepted        coverage=0.87  format=0.63  grounding=0.97  invariance=0.92  keys=0.57  length=1.00  text=1.00  tools=0.62  visual=0.94  ← выбрана
bge-m3|a=0.2|len=1.0|keys=1.0|accepted        coverage=0.87  format=0.63  grounding=0.97  invariance=0.92  keys=0.64  length=1.00  text=1.00  tools=0.50  visual=0.94
